## Creating Vaults with Claude Managed Agents

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")
github_access_token = os.getenv("GITHUB_ACCESS_TOKEN")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=anthropic_api_key)

### Creating the Vault

In [ ]:
vault = client.beta.vaults.create(
    display_name="GitHub Vault",
    metadata={
        "github_username": "debabrata-rana",
    }
)

print(vault.id)

### Registering the GitHub Access Token in the Vault

In [ ]:
credential = client.beta.vaults.credentials.create(
    vault_id=vault.id,
    display_name="GitHub Credential",
    auth={
        "type": "static_bearer",
        "mcp_server_url": "https://api.githubcopilot.com/mcp/",
        "token": github_access_token
    }
)

### Creating the Agent

In [ ]:
agent = client.beta.agents.create(
    name="GitHub Agent",
    model=claude_model,
    system="""
            You are an expert GitHub assistant.
            """,
    mcp_servers=[
        {
            "type": "url",
            "name": "GitHub",
            "url": "https://api.githubcopilot.com/mcp/"
        }
    ],
    tools=[
        {
            "type": "agent_toolset_20260401"
        },
        {
            "type": "mcp_toolset",
            "mcp_server_name": "GitHub",
            "default_config": {
                "permission_policy": {
                    "type": "always_allow"
                }
            }
        }
    ]
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### Creating the Environment

In [ ]:
environment = client.beta.environments.create(
    name="GitHub Agent Environment",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

### Creating the Session

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="GitHub Agent Session",
    vault_ids=[vault.id]
)

### Executing the Agent

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
    # Send the user message after the stream opens
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": """Give the contents of the GitHub Repository - debabrata-rana/Claude_Managed_Agents_Practice.
                                Please provide the file names and their contents.""",
                    },
                ],
            },
        ],
    )

    # Process streaming events
    for event in stream:
        match event.type:
            case "agent.message":
                for block in event.content:
                    print(block.text, end="")
            case "agent.tool_use":
                print(f"\n[Using tool: {event.name}]")
            case "session.status_idle":
                print("\n\nAgent finished.")
                break